# Feynman diagrams for Lindblad master equations — computational companion

Every numerical result in the paper and the Supplemental Material is reproduced here.
Each task states the problem, explains the diagrammatic set-up, runs the calculation
against an independent reference, and ends with the figure or table that appears in the
paper.

**Conventions.** Schrödinger-picture generators use $\mathcal L(\rho)=-i[H,\rho]+\mathcal D(\rho)$;
adjoint (Heisenberg) generators use $\mathcal L^\dagger(O)=+i[H,O]+\mathcal D^\dagger(O)$.
Both appear in `code/`; the plus sign is correct **only** for the adjoint. Every tilted
(counting-field) calculation is checked against the identity $c_1=\kappa\langle a^\dagger a\rangle$,
which is violated if that sign is wrong.

**Runtime.** `FAST = True` runs everything in a few minutes at reduced orders and cutoffs.
`FAST = False` reproduces the paper's numbers exactly (about an hour, dominated by the
exact references used only for validation). Heavy exact results are cached in `data/`.

In [ ]:
import sys, os, json, time
sys.path.insert(0, '../engines')
import numpy as np
import matplotlib.pyplot as plt
from math import comb
FAST = True
plt.rcParams.update({"font.family": "serif", "mathtext.fontset": "cm", "font.size": 9,
                     "legend.frameon": False, "xtick.direction": "in", "ytick.direction": "in"})
COL = ["#d95f02", "#1b9e77", "#7570b3", "#e7298a", "#66a61e", "#a6761d"]
from engine import Model, cav, add, scale, mul, destroy, steady
def mode_ops(alpha, d=1):
    I = np.eye(d); b, bd, one = cav(0,1,I), cav(1,0,I), cav(0,0,I)
    return add(b, scale(alpha, one)), add(bd, scale(np.conj(alpha), one)), b, bd
def moment_series(mod, m, n, alpha, N):
    tot = np.zeros(N+1, complex)
    for j in range(m+1):
        for k in range(n+1):
            tot += comb(m,j)*comb(n,k)*np.conj(alpha)**(m-j)*alpha**(n-k)*mod.series(cav(j,k,np.eye(1)), N)
    return tot
print("ready")

## 0. Engine validation

Two checks that everything downstream rests on.

**(a) Against an independent Schrödinger-picture recursion.** The coefficients of
$\langle a^\dagger a\rangle=\sum_N c_N U^N$ are recomputed as
$\rho^{(N)}=-\mathcal L_0^{+}\mathcal L_{\rm int}\rho^{(N-1)}$ in a truncated Fock basis:
different picture, different basis, no labels, no vertex table, no displaced frame.

**(b) The inverse Wick map** of Sec. IV F, which turns eigenoperator moments into
physical ones, checked on a squeezed-thermal steady state.

In [ ]:
from engine import liouvillian
import scipy.sparse as sps, scipy.sparse.linalg as spl
kappa, Dc, eta = 1.0, -1.0, 1.0
z = kappa/2 + 1j*Dc; alpha = eta/z
a, ad, b, bd = mode_ops(alpha)
mod = Model(1, z, lambda X: 0*X, np.eye(1), scale(0.5, mul(mul(ad,ad), mul(a,a))), 4)
Nchk = 8 if FAST else 16
c_diag = moment_series(mod, 1, 1, alpha, Nchk).real

Nc = 30 if FAST else 40
A = destroy(Nc); Ad = A.conj().T
H0 = Dc*Ad@A + 1j*(eta*Ad - np.conj(eta)*A)
L0 = liouvillian(H0, [np.sqrt(kappa)*A]); L1 = liouvillian(0.5*Ad@Ad@A@A, [])
rho0 = steady(H0, [np.sqrt(kappa)*A]).reshape(-1, order='F')
tr = np.eye(Nc).reshape(-1, order='F')
M = np.vstack([np.hstack([L0, tr[:,None]]), np.hstack([tr[None,:], [[0]]])])
lu = spl.splu(sps.csc_matrix(M))
c_rec = [np.trace(Ad@A@rho0.reshape(Nc,Nc,order='F')).real]; r = rho0
for N in range(1, Nchk+1):
    sol = lu.solve(np.concatenate([-(L1@r), [0]])); r = sol[:-1]
    c_rec.append(np.trace(Ad@A@r.reshape(Nc,Nc,order='F')).real)
print(" N   diagrams          recursion         rel. diff")
for N in range(0, Nchk+1, max(1, Nchk//6)):
    print(f"{N:3d}  {c_diag[N]: .8e}  {c_rec[N]: .8e}  {abs(c_diag[N]/c_rec[N]-1):.1e}")
import check_inversion; print()
import subprocess
subprocess.run([sys.executable, '../engines/check_inversion.py'], check=True)

## Task 1 — Driven Kerr cavity: loops, mean field, and an asymptotic series  (Fig. 4)

**Problem.** $H=\Delta_c a^\dagger a+i(\eta a^\dagger-\eta^*a)+\tfrac U2 (a^\dagger)^2a^2$ with
$\kappa\mathcal D[a]$, $\Delta_c=-\kappa$, $\eta=\kappa$.

**Set-up.** The drive is removed exactly by $b=a-\alpha$, $\alpha=\eta/z$; $\mathcal L_0$ is the
damped mode with a vacuum steady state and the Kerr term is the perturbation. Mean-field
theory is the sum of all loop-free contributions, so the gap between the two curves is the
loop content. Panel (b) shows the asymptotic character: the error falls to a minimum at an
order that shrinks as $U$ grows, so the paper quotes every series at a stated, fixed order.

In [ ]:
Nmax = 12 if FAST else 16
cn = moment_series(mod, 1, 1, alpha, Nmax).real
def kerr_exact(U, Nc=30):
    A = destroy(Nc); Ad = A.conj().T
    H = Dc*Ad@A + 1j*(eta*Ad - np.conj(eta)*A) + U/2*Ad@Ad@A@A
    rho = steady(H, [np.sqrt(kappa)*A]); return np.trace(Ad@A@rho).real
def kerr_meanfield(U):
    r = np.roots([U**2, 2*Dc*U, Dc**2+kappa**2/4, -abs(eta)**2]) if U>0 else [abs(alpha)**2]
    r = np.array(r); return float(np.min(r[np.abs(np.imag(r))<1e-9].real))
Us = np.linspace(0, 0.3, 21 if FAST else 61)
ex = np.array([kerr_exact(U) for U in Us]); mf = np.array([kerr_meanfield(U) for U in Us])
pw = Us[:,None]**np.arange(Nmax+1)[None,:]
fig,(ax1,ax2) = plt.subplots(2,1,figsize=(4.2,5.6))
ax1.plot(Us, ex, 'k-', lw=1.8, label='exact'); ax1.plot(Us, mf, '--', color='0.5', label='mean field ($L=0$)')
for c,N in zip(COL,[1,2,4,8]): ax1.plot(Us,(pw[:,:N+1]*cn[None,:N+1]).sum(1),color=c,lw=1,label=f'$N\\leq{N}$')
ax1.set_ylim(0.6,3.2); ax1.set_xlabel('$U/\\kappa$'); ax1.set_ylabel(r'$\langle a^\dagger a\rangle$'); ax1.legend(fontsize=7)
for c,U in zip(COL,[0.05,0.1,0.2,0.3]):
    e = kerr_exact(U); err = np.abs(np.cumsum(cn*U**np.arange(Nmax+1))-e)
    ax2.semilogy(range(Nmax+1), err, 'o-', ms=3, color=c, label=f'$U/\\kappa={U}$')
    k = int(np.argmin(err)); print(f"U={U}: error minimum at N={k} ({err[k]:.2e}), error at N={Nmax}: {err[Nmax]:.2e}")
ax2.set_xlabel('order $N$'); ax2.set_ylabel('absolute error'); ax2.legend(fontsize=7)
fig.tight_layout(); fig.savefig('out_fig4_kerr.pdf'); plt.show()

## Task 2 — Finite time: switching on the drive  (SM)

At finite time every diagram contributes, not only those closing on $r_{00}$: a diagram with
outgoing label $O_{pq}$ closes on the initial-state component $r_{qp}$ with weight
$\mathrm{Tr}[O_{pq}\rho_0]$. The cavity starts empty, which in the displaced frame is a
coherent state of amplitude $-\alpha$, so every state label contributes.

In [ ]:
from finite_time import kerr_finite_time, kerr_exact_time
U = 0.1; ts = np.linspace(0,10,41 if FAST else 101); Nft = 6 if FAST else 8
ser, parts = kerr_finite_time(kappa, Dc, eta, U, ts, Nft, lambda a_,ad_: mul(ad_,a_), by_degree=True)
ext = kerr_exact_time(kappa, Dc, eta, U, ts); ps = np.cumsum(ser.real, axis=1)
fig,(ax1,ax2) = plt.subplots(2,1,figsize=(4.2,5.6),sharex=True)
ax1.plot(ts, ext,'k-',lw=1.8,label='exact')
for c,N in zip(COL,[0,1,2,4,Nft]): ax1.plot(ts, ps[:,N], color=c, lw=1, label=f'$N\\leq{N}$')
ax1.set_ylabel(r'$\langle a^\dagger a\rangle(t)$'); ax1.legend(fontsize=7)
for c,d in zip(COL,range(5)):
    if d in parts[1]: ax2.plot(ts, parts[1][d].real, color=c, label=('$r_{00}$' if d==0 else f'{d} outgoing'))
ax2.plot(ts, ser[:,1].real,'k--',label='total, $N=1$'); ax2.axhline(0,color='0.7',lw=0.5)
ax2.set_xlabel(r'$\kappa t$'); ax2.set_ylabel('first-order correction'); ax2.legend(fontsize=7)
fig.tight_layout(); fig.savefig('out_kerr_finite_time.pdf'); plt.show()
print(f"max |error| through N={Nft}: {np.max(np.abs(ps[:,Nft]-ext)):.2e}")

## Task 3 — Cubic nonlinearity, and a cutoff pathology in the reference  (SM)

$H_{\rm int}=-\tfrac{ig}{2}[(a^\dagger)^2a-a^\dagger a^2]$ with $\gamma=2.108$, $\omega=2.153$,
$f=1.17+4.617i$. Above $g\approx0.62$ the truncated-Fock steady state itself stops converging
in the cutoff, so the two "exact" curves separate while the diagrams track the converged region.

In [ ]:
from sparse_ss import steady_sparse
gam, om, f = 2.108, 2.153, 1.17+4.617j
zc = gam/2+1j*om; etac = f/2; alc = etac/zc
a3, ad3, _, _ = mode_ops(alc)
modc = Model(1, zc, lambda X: 0*X, np.eye(1),
             scale(-0.5j, add(mul(mul(ad3,ad3),a3), scale(-1, mul(ad3, mul(a3,a3))))), 3)
Nmc = 12 if FAST else 24
sa = moment_series(modc, 0, 1, alc, Nmc)
gs = np.linspace(0, 0.8, 17 if FAST else 41); cut = (30,50) if FAST else (40,80)
exc = {}
for Ncut in cut:
    A_ = destroy(Ncut); Ad_ = A_.conj().T; v=[]
    for g in gs:
        H = om*Ad_@A_ + 1j*(etac*Ad_-np.conj(etac)*A_) + g*(-0.5j)*(Ad_@Ad_@A_-Ad_@A_@A_)
        v.append(np.trace(A_@steady_sparse(H,[np.sqrt(gam)*A_])))
    exc[Ncut]=np.array(v)
fig,(ax1,ax2)=plt.subplots(2,1,figsize=(4.2,5.6),sharex=True)
for ax,fn,lab in [(ax1,np.real,r'Re$\langle a\rangle$'),(ax2,np.imag,r'Im$\langle a\rangle$')]:
    ax.plot(gs,fn(exc[cut[1]]),'k-',lw=1.8,label=f'exact, $n_{{\\max}}={cut[1]}$')
    ax.plot(gs,fn(exc[cut[0]]),':',color='0.4',lw=1.2,label=f'exact, $n_{{\\max}}={cut[0]}$')
    for c,N in zip(COL,[1,2,4,Nmc]):
        ax.plot(gs,fn([np.sum(sa[:N+1]*g**np.arange(N+1)) for g in gs]),color=c,lw=1,label=f'$N\\leq{N}$')
    ax.set_ylabel(lab)
ax1.legend(fontsize=6,ncol=2); ax2.set_xlabel('coupling $g$'); ax1.set_ylim(0.6,1.1); ax2.set_ylim(0.15,1.0)
fig.tight_layout(); fig.savefig('out_cubic.pdf'); plt.show()
k=np.argmin(abs(gs-0.5)); print(f"g=0.5: exact {exc[cut[1]][k]:.6f}, series {np.sum(sa[:Nmc+1]*0.5**np.arange(Nmc+1)):.6f}")

## Task 4 — A two-level atom versus a classical dipole  (SM)

Atom and cavity resonant, laser detuned by $\Delta$ from both; $\kappa=\gamma$, $g=0.3\gamma$,
weak drive. The atom and a harmonic dipole of the same rates transmit the same intensity;
only the photon statistics distinguish them, and the diagrams reproduce the antibunching
that exclusion produces.

In [ ]:
from atom_cavity import series_cavity, exact_cavity
kap, gam_a, g_a, eta_a = 1.0, 1.0, 0.3, 0.02
Ds = np.linspace(-2,2,17 if FAST else 81); Nat = 12 if FAST else 32
obs = {"n": lambda a_,ad_: mul(ad_,a_), "n2": lambda a_,ad_: mul(mul(ad_,ad_),mul(a_,a_))}
exn,exg,pn,pg = [],[],[],[]
for D in Ds:
    s,al = series_cavity("atom",kap,gam_a,D,D,eta_a,g_a,Nat,obs)
    n,n2,_ = exact_cavity("atom",kap,gam_a,D,D,eta_a,g_a)
    cn_,cn2 = np.cumsum(s["n"].real), np.cumsum(s["n2"].real)
    exn.append(n/abs(al)**2); exg.append(n2/n**2); pn.append(cn_/abs(al)**2); pg.append(cn2/cn_**2)
exn,exg,pn,pg = map(np.array,(exn,exg,pn,pg))
zc_,za_ = kap/2+1j*Ds, gam_a/2+1j*Ds
Tcl = np.abs(zc_*za_/(zc_*za_+g_a**2))**2
fig,(ax1,ax2)=plt.subplots(2,1,figsize=(4.2,5.6),sharex=True)
ax1.plot(Ds,exn,'k-',lw=1.8,label='atom, exact'); ax1.plot(Ds,Tcl,'--',color='0.5',label='classical dipole')
for c,N in zip(COL,[2,4,8]): ax1.plot(Ds,pn[:,N],color=c,lw=1,label=f'atom, $N\\leq{N}$')
ax1.set_ylabel(r'$\langle a^\dagger a\rangle/|\alpha|^2$'); ax1.legend(fontsize=7)
ax2.axhline(1,ls='--',color='0.5',label='classical dipole'); ax2.plot(Ds,exg,'k-',lw=1.8,label='atom, exact')
for c,N in zip(COL[1:],[4,8,Nat]): ax2.plot(Ds,pg[:,N],color=c,lw=1,label=f'atom, $N\\leq{N}$')
ax2.set_ylim(0.85,1.07); ax2.set_xlabel(r'$\Delta/\gamma$'); ax2.set_ylabel('$g^{(2)}(0)$'); ax2.legend(fontsize=7,ncol=2)
fig.tight_layout(); fig.savefig('out_atom_cavity.pdf'); plt.show()
k=np.argmin(abs(Ds)); print(f"resonance: exact g2 {exg[k]:.5f}, diagrams {pg[k,Nat]:.5f}")

## Task 5 — Driven atom: bare versus dressed lines, and the Mollow triplet  (Fig. 8, 9)

*Bare split*: the drive is a vertex; the drive vertices form a geometric ladder with ratio
$-\Omega^2/2D$, so the series diverges beyond $\Omega=\gamma/\sqrt2$. *Dressed split*: the
driven atom sits inside $\mathcal L_0$ and only the exchange is perturbative; each of the three
dressed eigenoperators produces one Mollow component.

In [ ]:
sm = np.array([[0,0],[1,0]],complex); sp_ = sm.conj().T
sz = np.diag([1.,-1.]).astype(complex); sx = sm+sp_; I2 = np.eye(2)
def atom_gen(Da,Om,gam):
    Ha = Da/2*sz + Om/2*sx
    La = lambda X: 1j*(Ha@X-X@Ha) + gam*(sp_@X@sm - 0.5*(sp_@sm@X + X@sp_@sm))
    return La, steady(Ha,[np.sqrt(gam)*sm])
def em_series(kap,Dc_,Da,Om,gam,g,N,O,dressed=True):
    z_ = kap/2+1j*Dc_; A_,Ad_ = cav(0,1,I2), cav(1,0,I2)
    Hx = scale(g, add(mul(Ad_,cav(0,0,sm)), mul(A_,cav(0,0,sp_))))
    if dressed: La,rho = atom_gen(Da,Om,gam); Hint = Hx
    else:       La,rho = atom_gen(Da,0.0,gam); Hint = add(Hx, cav(0,0,Om/2*sx))
    return Model(2,z_,La,rho,Hint,2).series(O,N)
def em_exact(kap,Dc_,Da,Om,gam,g,Nc_=8):
    A_=np.kron(destroy(Nc_),I2); Ad_=A_.conj().T; S=np.kron(np.eye(Nc_),sm)
    H=Dc_*Ad_@A_+np.kron(np.eye(Nc_),Da/2*sz+Om/2*sx)+g*(Ad_@S+A_@S.conj().T)
    return np.trace(Ad_@A_@steady(H,[np.sqrt(kap)*A_,np.sqrt(gam)*S])).real
gam_,kap_ = 1.0, 0.5
Dcs = np.linspace(-8,8,33 if FAST else 161)
ex_a=np.array([em_exact(kap_,D,0.0,5.0,gam_,0.2) for D in Dcs])
n2_a=np.array([np.sum(em_series(kap_,D,0.0,5.0,gam_,0.2,2,cav(1,1,I2))) for D in Dcs]).real
n4_a=np.array([np.sum(em_series(kap_,D,0.0,5.0,gam_,0.2,4,cav(1,1,I2))) for D in Dcs]).real
Oms=np.linspace(0.05,2.0,25 if FAST else 100); g_=0.1
def n_cl(O):
    M=np.array([[kap_/2,1j*g_],[1j*g_,gam_/2]]); av,_=np.linalg.solve(M,[0,-1j*O/2]); return abs(av)**2
ncl=np.array([n_cl(O) for O in Oms])
ex_b=np.array([em_exact(kap_,0.0,0.0,O,gam_,g_) for O in Oms])/ncl
nd4=np.array([np.sum(em_series(kap_,0.0,0.0,O,gam_,g_,4,cav(1,1,I2))) for O in Oms]).real/ncl
Nb=8 if FAST else 16
sb=np.array([np.cumsum(em_series(kap_,0.0,0.0,O,gam_,g_,Nb,cav(1,1,I2),dressed=False)).real for O in Oms])/ncl[:,None]
fig,(ax1,ax2)=plt.subplots(2,1,figsize=(4.2,5.6))
ax1.plot(Dcs,1e3*ex_a,'k-',lw=1.8,label='exact'); ax1.plot(Dcs,1e3*n2_a,'--',color=COL[0],label='dressed $N\\leq2$')
ax1.plot(Dcs,1e3*n4_a,':',color=COL[1],lw=1.5,label='dressed $N\\leq4$')
ax1.set_xlabel(r'$\Delta_c/\gamma$'); ax1.set_ylabel(r'$10^3\langle a^\dagger a\rangle$'); ax1.legend(fontsize=7)
ax1.set_title('Mollow triplet seen by the cavity',fontsize=9)
ax2.axvspan(gam_/np.sqrt(2),2,color='0.93',lw=0); ax2.axhline(1,ls='--',color='0.5',label='classical dipole')
ax2.plot(Oms,ex_b,'k-',lw=1.8,label='atom, exact')
for c,N in zip(COL[2:],[4,Nb]): ax2.plot(Oms,sb[:,N],color=c,lw=1,label=f'bare $N\\leq{N}$')
ax2.plot(Oms,nd4,':',color=COL[1],lw=1.5,label='dressed $N\\leq4$')
ax2.set_ylim(0,1.3); ax2.set_xlabel(r'$\Omega/\gamma$'); ax2.set_ylabel(r'$\langle a^\dagger a\rangle/\langle a^\dagger a\rangle_{\rm cl}$')
ax2.legend(fontsize=7,ncol=2)
fig.tight_layout(); fig.savefig('out_emitter.pdf'); plt.show()
print(f"max relative error, dressed N<=4 (panel a): {np.max(np.abs(n4_a/ex_a-1)):.4f}")

## Task 6 — Kerr ring: polynomial versus exponential cost  (Fig. 6)

$K$ driven Kerr cavities on a ring. The linear ring is Gaussian: a uniform displacement plus
a Fourier transform diagonalizes $\mathcal L_0$ once for every $K$, and the Kerr term becomes a
momentum-conserving vertex. Exact steady states are available only up to $K\approx4$.

In [ ]:
import multimode as MM
from ring_ss_evolve import ring_steady_evolve
P = dict(kappa=1.0, Delta=-1.0, J=0.4, eta=1.0)
r1 = MM.Ring(K=1, kappa=1.0, Delta=-1.0, J=0.0, eta=1.0, U=0.15)
a0,ad0 = r1.site_ops(0)
z1 = 0.5-1j; al1 = 1/z1
A1,Ad1,_,_ = mode_ops(al1)
mod1 = Model(1, z1, lambda X: 0*X, np.eye(1), scale(0.15/2, mul(mul(Ad1,Ad1),mul(A1,A1))), 4)
print("K=1 cross-check   multimode:", np.round(np.cumsum(r1.series(MM.mul(ad0,a0),4)).real,8))
print("                 single mode:", np.round(np.cumsum(moment_series(mod1,1,1,al1,4)).real,8))
Us = np.linspace(0,0.3,13); exU = json.load(open('data/ring_exactU.json'))
g2_ser = []
for U in Us:
    r = MM.Ring(K=3,U=U,**P); b0,bd0 = r.site_ops(0)
    n = np.cumsum(r.series(MM.mul(bd0,b0),2)).real
    n2 = np.cumsum(r.series(MM.mul(MM.mul(bd0,bd0),MM.mul(b0,b0)),2)).real
    g2_ser.append(n2/n**2)
g2_ser = np.array(g2_ser)
exG = np.array([exU[f"{U:.3f}"]["n0n0"]/exU[f"{U:.3f}"]["n0"]**2 for U in Us])
Ks = [3,4] if FAST else [3,4,5,6]; td=[]
for K in Ks:
    t=time.time(); r=MM.Ring(K=K,U=0.15,**P); b0,bd0=r.site_ops(0); r.series(MM.mul(bd0,b0),2); td.append(time.time()-t)
te = {}
for K,nm in [(2,8),(3,6)]:
    t=time.time(); ring_steady_evolve(K,U=0.15,nmax=nm,**P); te[K]=time.time()-t
paper_exact = json.load(open('data/ring_timing.json'))["exact"]
fig,(ax1,ax2)=plt.subplots(2,1,figsize=(4.2,5.6))
ax1.plot(Us,exG,'k-',lw=1.8,label='exact ($K=3$)')
for c,N,ls in zip(COL,[0,1,2],['-','--',':']): ax1.plot(Us,g2_ser[:,N],color=c,ls=ls,label=f'$N\\leq{N}$')
ax1.set_xlabel('$U/\\kappa$'); ax1.set_ylabel('on-site $g^{(2)}(0)$'); ax1.legend(fontsize=7)
Kall=np.arange(2,11)
ax2.semilogy(Kall, float(paper_exact["4"])*5.0**(2*(Kall-4)),':',color=COL[0],label=r'$\propto n_{\max}^{2K}$')
ax2.semilogy([int(k) for k in paper_exact],[float(v) for v in paper_exact.values()],'s-',color=COL[0],label='exact (paper run)')
ax2.semilogy(list(te),list(te.values()),'s',mfc='none',color=COL[0],label='exact (this run)')
ax2.semilogy(Ks,td,'o-',color=COL[1],label='diagrams $N\\leq2$ (this run)')
ax2.set_ylim(1e-2,1e9); ax2.set_xlabel('ring size $K$'); ax2.set_ylabel('runtime (s)'); ax2.legend(fontsize=7)
fig.tight_layout(); fig.savefig('out_ring.pdf'); plt.show()
print("g2 error at U=0.15, N<=2:", round(abs(g2_ser[6,2]-exG[6]),5))

## Task 7 — Against the second-order cumulant closure  (Fig. 7)

The closure (Hartree–Fock–Bogoliubov) is the scalable competitor. Below $U\approx0.15\kappa$
the diagrams have one to three orders of magnitude smaller error; approaching the mean-field
bistability neither method is reliable, but the closure fails silently while the diagrammatic
terms turn over.

In [ ]:
from cumulant import kerr_cumulant_ss
cn2 = moment_series(mod, 2, 2, alpha, Nmax).real
Uc = np.linspace(0.02, 0.3, 15 if FAST else 29)
E = {k: [] for k in ['mf','hfb','N2','N4','N8']}; G = {k: [] for k in ['hfb','N2','N4','N8']}
x0 = None
for U in Uc:
    n_ex = kerr_exact(U)
    A_=destroy(30); Ad_=A_.conj().T
    H=Dc*Ad_@A_+1j*(eta*Ad_-np.conj(eta)*A_)+U/2*Ad_@Ad_@A_@A_
    rho=steady(H,[np.sqrt(kappa)*A_]); g_ex=np.trace(Ad_@Ad_@A_@A_@rho).real/n_ex**2
    pw_=U**np.arange(Nmax+1); pn_=np.cumsum(cn*pw_); pg_=np.cumsum(cn2*pw_)/pn_**2
    c=kerr_cumulant_ss(kappa,Dc,eta,U,x0=x0); x0=c['x']
    E['mf'].append(abs(kerr_meanfield(U)/n_ex-1)); E['hfb'].append(abs(c['n']/n_ex-1))
    E['N2'].append(abs(pn_[2]/n_ex-1)); E['N4'].append(abs(pn_[4]/n_ex-1)); E['N8'].append(abs(pn_[8]/n_ex-1))
    G['hfb'].append(abs(c['g2']/g_ex-1)); G['N2'].append(abs(pg_[2]/g_ex-1)); G['N4'].append(abs(pg_[4]/g_ex-1)); G['N8'].append(abs(pg_[8]/g_ex-1))
sty={'mf':('mean field','0.5','--'),'hfb':('2nd-order cumulant',COL[0],'-'),'N2':('diagrams $N\\leq2$',COL[2],':'),
     'N4':('diagrams $N\\leq4$',COL[1],'-.'),'N8':('diagrams $N\\leq8$','k','-')}
fig,(ax1,ax2)=plt.subplots(2,1,figsize=(4.2,5.6),sharex=True)
for k in E: ax1.semilogy(Uc,np.maximum(E[k],1e-7),color=sty[k][1],ls=sty[k][2],label=sty[k][0])
for k in G: ax2.semilogy(Uc,np.maximum(G[k],1e-7),color=sty[k][1],ls=sty[k][2],label=sty[k][0])
for ax in (ax1,ax2): ax.axvspan(0.16,0.3,color='0.93',lw=0); ax.set_ylim(1e-6,1)
ax1.set_ylabel(r'rel. error, $\langle a^\dagger a\rangle$'); ax2.set_ylabel('rel. error, $g^{(2)}(0)$')
ax2.set_xlabel('$U/\\kappa$'); ax1.legend(fontsize=6)
fig.tight_layout(); fig.savefig('out_compare.pdf'); plt.show()
k=np.argmin(abs(Uc-0.2)); print(f"U=0.2: closure error on n = {E['hfb'][k]:.3f} (converged, no internal warning)")

## Task 8 — Disorder and site-resolved correlations  (Fig. 8 of the main text)

With site-dependent detunings the drift is still normal, so the construction is unchanged: the
momentum modes become numerically diagonalized normal modes and the vertex becomes a dense
tensor. The engine prunes, at every vertex, every degree sector the remaining vertices cannot
bring back to the steady state.

In [ ]:
import tensor_engine as TE
from hfb_chain import hfb_chain
r3 = MM.Ring(K=3,U=0.15,**P); b0,bd0 = r3.site_ops(0)
ch3 = TE.Chain(3,1.0,[-1.0]*3,0.4,1.0,0.15)
a3_,ad3_ = ch3.site_a(0), ch3.site_ad(0)
print("K=3 uniform cross-check  momentum:", np.round(np.cumsum(r3.series(MM.mul(bd0,b0),2)).real,6))
print("                           tensor:", np.round(np.cumsum(ch3.series(TE.mul(ad3_,a3_),2)).real,6))
rng = np.random.default_rng(7); K = 10; det = -1.0+0.3*rng.uniform(-1,1,K); U = 0.15
ch = TE.Chain(K,1.0,det,0.4,1.0,U); h,_ = hfb_chain(K,1.0,det,0.4,1.0,U)
Nord = 2 if FAST else 4
n_,g2_,g3_ = [],[],[]
t0=time.time()
for j in range(K):
    a_,ad_ = ch.site_a(j), ch.site_ad(j)
    n = np.cumsum(ch.series(TE.mul(ad_,a_),Nord)).real
    n2 = np.cumsum(ch.series(TE.mul(TE.mul(ad_,ad_),TE.mul(a_,a_)),Nord)).real
    n3 = np.cumsum(ch.series(TE.mul(TE.mul(TE.mul(ad_,ad_),ad_),TE.mul(TE.mul(a_,a_),a_)),Nord)).real
    n_.append(n[Nord]); g2_.append(n2[Nord]/n[Nord]**2); g3_.append(n3[Nord]/n[Nord]**3)
print(f"K={K}, order {Nord}: {time.time()-t0:.0f}s")
sites=np.arange(K)
fig,(ax0,ax1,ax2)=plt.subplots(3,1,figsize=(4.2,6.0),sharex=True)
ax0.bar(sites,n_,color='0.75',width=0.6,label=r'$\langle a_j^\dagger a_j\rangle$'); ax0.plot(sites,-det,'k.',ms=4,label=r'$-\Delta_j/\kappa$')
ax0.set_ylabel('occupation'); ax0.legend(fontsize=7)
ax1.plot(sites,[h[j]['g2'] for j in sites],'s-',color=COL[0],ms=3.5,label='2nd-order cumulant')
ax1.plot(sites,g2_,'o-',color=COL[1],ms=3.5,label=f'diagrams $N\\leq{Nord}$')
ax1.set_ylabel('$g^{(2)}_j(0)$'); ax1.legend(fontsize=7)
ax2.plot(sites,[h[j]['g3'] for j in sites],'s-',color=COL[0],ms=3.5,label='2nd-order cumulant')
ax2.plot(sites,g3_,'o-',color=COL[1],ms=3.5,label=f'diagrams $N\\leq{Nord}$')
ax2.set_ylabel('$g^{(3)}_j(0)$'); ax2.set_xlabel('site $j$'); ax2.set_xticks(sites); ax2.legend(fontsize=7)
fig.tight_layout(); fig.savefig('out_disorder.pdf'); plt.show()

## Task 9 — Counting statistics  (Fig. 10, Table V)

The counting-field-tilted generator only raises the label in the displaced frame, so it is
upper triangular: the spectrum is $\Lambda^\chi_{mn}=\Lambda_{mn}+\kappa|\alpha|^2(e^\chi-1)$,
the propagator is unchanged, and the left eigenoperator $Y_{00}$ is the resummed ladder.
The interacting corrections are the same diagrams with $Y_{00}$ as external operator, with the
spectral projection removing the $Y_{00}$ **direction** (not the $O_{00}$ label).

Reference methods: mean field (Poissonian), the Gaussian/HFB model with its own tilted
statistics, and the exact tilted Liouvillian. Every tilted calculation is checked against
$c_1=\kappa\langle a^\dagger a\rangle$.

In [ ]:
from fcs import theta_series
from tilted_exact import theta_lead, cumulants
from gaussian_fcs import gaussian_model
rows=[]
for U in [0.05,0.10]:
    Ncf=26 if not FAST else 20
    A_=destroy(Ncf); Ad_=A_.conj().T
    Hx=Dc*Ad_@A_+1j*(eta*Ad_-np.conj(eta)*A_)+U/2*Ad_@Ad_@A_@A_
    ce=cumulants(lambda x: theta_lead(Hx,[A_],[kappa],[x]))
    rho=steady(Hx,[np.sqrt(kappa)*A_]); nss=np.trace(Ad_@A_@rho).real
    Hg,ag,_,p=gaussian_model(kappa,Dc,eta,U,Nc=Ncf)
    cg=cumulants(lambda x: theta_lead(Hg,[ag],[kappa],[x]))
    cd=cumulants(lambda x: np.cumsum(theta_series(kappa,Dc,eta,U,x,6 if FAST else 8)).real[-1])
    cmf=(kappa*abs(p['alpha'])**2,1.0,1.0)
    rows.append((U,ce,cmf,cg,cd,nss))
    print(f"U={U}:  c1 identity check: exact c1 {ce[0]:.5f} vs kappa*n {kappa*nss:.5f}")
    for name,c in [("exact",ce),("mean field",cmf),("Gaussian FCS",cg),("diagrams",cd)]:
        print(f"   {name:<14} c1 {c[0]:8.5f}  Fano {c[1]:8.5f}  c3/c1 {c[2]:8.5f}")
    print()
k3=json.load(open('data/k3_theta.json'))
if len(k3)==5:
    h=0.05; th=np.array([k3[f"{x:.3f}"] for x in [-2*h,-h,0.0,h,2*h]])
    c1=(th[3]-th[1])/(2*h); c2=(th[3]-2*th[2]+th[1])/h**2; c3=(th[4]-2*th[3]+2*th[1]-th[0])/(2*h**3)
    print(f"K=3 exact (cached, n_max=9): c1 {c1:.5f}  Fano {c2/c1:.5f}  c3/c1 {c3/c1:.5f}")
from fcs_multi import TiltedChain, cumulants as cum_multi
det3=[-1.0]*3; eta3=np.array([1.0,0.0,0.0])
def build(chi,U=0.05):
    ch=TiltedChain(3,1.0,det3,0.4,eta3,U,ring=True); ch.set_tilt(0,chi,1.0); return ch
for N,md_ in [(2,4),(3,5)] if FAST else [(2,4),(3,5),(4,6)]:
    c=cum_multi(build,None,N,md_,h=0.05)
    print(f"K=3 diagrams N<={N}: c1 {c[0]:.5f}  Fano {c[1]:.5f}  c3/c1 {c[2]:.5f}")

### Counting cumulants across a disordered chain (Fig. 10)

In [ ]:
rngf=np.random.default_rng(5); [rngf.uniform(-1,1,k) for k in (2,4,6)]
Kf=8; detf=-1.0+0.3*rngf.uniform(-1,1,Kf); etaf=np.zeros(Kf); etaf[0]=1.0
Us_f=[0.0,0.04,0.08,0.10] if FAST else [0.0,0.02,0.04,0.06,0.08,0.10]
fano,c3r=[],[]
for U in Us_f:
    def bf(chi,U=U):
        ch=TiltedChain(Kf,1.0,detf,0.4,etaf,U,ring=False); ch.set_tilt(0,chi,1.0); return ch
    c=cum_multi(bf,None,3 if U>0 else 0,5 if U>0 else 2,h=0.05)
    fano.append(c[1]); c3r.append(c[2]); print(U, np.round(c,5))
fig,ax=plt.subplots(figsize=(4.2,2.8))
ax.axhline(1.0,color='0.5',ls=(0,(4,2)),lw=1.2,label='mean field (Poissonian)')
ax.plot(Us_f,fano,'o-',color=COL[0],ms=3.5,label=r'Fano $c_2/c_1$')
ax.plot(Us_f,c3r,'s-',color=COL[1],ms=3.5,label=r'$c_3/c_1$')
ax.set_xlabel('$U/\\kappa$'); ax.set_ylabel('counting cumulant ratio'); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig('out_fcs.pdf'); plt.show()

## Summary

| Task | Content | Independent reference |
|---|---|---|
| 0 | engine validation | Schrödinger-picture recursion; Wick inversion |
| 1 | Kerr cavity, loops, asymptotic series | exact steady state |
| 2 | finite time, switch-on | exact time evolution |
| 3 | cubic nonlinearity | exact at two Fock cutoffs |
| 4 | atom vs classical dipole | exact steady state |
| 5 | driven atom, Mollow triplet | exact steady state |
| 6 | Kerr ring, cost scaling | exact up to $K=4$ |
| 7 | vs. second-order cumulant closure | exact steady state |
| 8 | disordered chain, $g^{(2)}$, $g^{(3)}$ | HFB closure; $K=3$ exact |
| 9 | counting statistics | exact tilted Liouvillian, $K=1$ and $K=3$ |

Known limits, as stated in the paper: the expansion requires $U|\alpha|^2\ll\kappa$; near the
mean-field bistability neither the diagrams nor the closure is reliable; the counting expansion
holds for small $\chi$, the large-$\chi$ branch switch being a dynamical transition the
perturbative branch does not follow.